# TASK 1. PROJECT OVERVIEW AND KEY LEARNING OBJECTIVES

This is the **Gemini version** of the original OpenAI Multi-AI Agents Workflow notebook.  
Every cell is a line-by-line equivalent using **Google's free Gemini model** instead of OpenAI GPT.

### Quick Reference: OpenAI → Gemini Mapping

| Original (OpenAI) | Gemini Equivalent |
|---|---|
| `pip install openai-agents` | `pip install google-generativeai` |
| `from openai import OpenAI` | `import google.generativeai as genai` |
| `from agents import Agent, Runner, function_tool, SQLiteSession` | Custom helpers + `genai.GenerativeModel` |
| `OpenAI(api_key=...)` | `genai.configure(api_key=...)` |
| `model="gpt-4.1-mini"` | `model_name="models/gemini-2.0-flash"` (free) |
| `Agent(name, instructions, model, tools, output_type)` | `genai.GenerativeModel(...)` + JSON schema |
| `@function_tool` decorator | Plain Python function (Gemini auto-generates schema) |
| `output_type = AnalysisSummary` (Pydantic) | JSON output via `response_mime_type` + `response_schema` |
| `SQLiteSession` (memory) | `conversation_history` list (in-memory) |
| `await Runner.run(agent, input, session)` | `run_agent()` helper with function-call loop |

# TASK 2. SETUP API KEYS & DEFINE REQUIRED TOOLS

Instead of OpenAI, we use **Google's Gemini API** (free tier).

- **Gemini API Key:** [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- **Tavily API Key:** [https://tavily.com/](https://tavily.com/)

Create a `.env` file:
```
GEMINI_API_KEY=your-gemini-api-key-here
TAVILY_API_KEY=tvly-YourSecretTavilyKey...
```

In [1]:
# ── Original Cell 3: pip install ─────────────────────────────────────────────
# ORIGINAL:  !pip install -q openai-agents==0.2.2 python-dotenv requests pydantic
# GEMINI:    Replace openai-agents with google-generativeai

%pip install -q google-generativeai python-dotenv requests pydantic

Note: you may need to restart the kernel to use updated packages.


In [6]:
# ── Original Cell 4: Load keys & imports ─────────────────────────────────────
# ORIGINAL:
#   import os
#   import requests
#   from IPython.display import display, Markdown
#   from openai import OpenAI
#   from dotenv import load_dotenv
#   from agents import Agent, Runner, function_tool, SQLiteSession
#   from typing_extensions import TypedDict
#   from pydantic import BaseModel
#   load_dotenv()
#   openai_api_key = os.getenv("OPENAI_API_KEY")
#   tavily_api_key = os.getenv("TAVILY_API_KEY")
#
# GEMINI EQUIVALENT:

import os
import json
import requests
from IPython.display import display, Markdown
import google.generativeai as genai            # Replaces: from openai import OpenAI
from dotenv import load_dotenv
# NOT needed: from agents import Agent, Runner, function_tool, SQLiteSession
# (We build equivalent functionality with plain Gemini SDK)
from typing_extensions import TypedDict
from pydantic import BaseModel

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")   # Replaces: os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# Configure Gemini SDK (replaces: OpenAI(api_key=openai_api_key))
genai.configure(api_key=gemini_api_key)

# ── IMPORTANT: Model name ────────────────────────────────────────────────────
# The legacy google-generativeai package REQUIRES the "models/" prefix.
# Without it you get: InvalidArgument: 400 unexpected model name format
# We define it once here and reuse it across all agents.
GEMINI_MODEL = "models/gemini-2.5-flash-lite"       # Replaces: "gpt-4.1-mini"

print("✅ API keys loaded")
print(f"  Gemini … {gemini_api_key[:5]}***")
print(f"  Tavily … {tavily_api_key[:5]}***")
print(f"  Model  … {GEMINI_MODEL}")

✅ API keys loaded
  Gemini … AIzaS***
  Tavily … tvly-***
  Model  … models/gemini-2.5-flash-lite


In [8]:
# ── Original Cell 5: Helper to print markdown ───────────────────────────────
# ORIGINAL:
#   def print_markdown(text):
#       display(Markdown(text))
#
# GEMINI EQUIVALENT: (identical — nothing model-specific)

def print_markdown(text):
    display(Markdown(text))

In [9]:
# ── Original Cell 6: Tavily search tool ──────────────────────────────────────
# ORIGINAL:
#   class TavilySearchParams(TypedDict):
#       query: str
#       max_results: int
#
#   @function_tool                              <-- OpenAI decorator (REMOVED)
#   def tavily_search(params: TavilySearchParams) -> str:
#       payload = {
#           "query": params["query"],           <-- accessed via dict key
#           "max_results": params.get("max_results", 3),
#       }
#
# GEMINI EQUIVALENT:
#   - No @function_tool decorator needed — Gemini reads function signature + docstring
#   - Parameters are direct keyword args instead of a TypedDict dict

# We keep the TypedDict for reference (not used by Gemini)
class TavilySearchParams(TypedDict):
    query: str
    max_results: int


# No @function_tool decorator needed for Gemini
def tavily_search(query: str, max_results: int = 3) -> str:
    """
    Searches the web using Tavily API and returns a summary of top results.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return (default 3).

    Returns:
        A formatted string summarizing the top search results.
    """
    # Tavily search endpoint
    url = "https://api.tavily.com/search"

    # Tell the API we're sending JSON
    headers = {"Content-Type": "application/json"}

    # Build the request body
    payload = {
        "api_key": tavily_api_key,
        "query": query,               # CHANGED: was params["query"]
        "max_results": max_results,   # CHANGED: was params.get("max_results", 3)
    }

    # Send the POST request
    response = requests.post(url, json=payload, headers=headers)
    if response.status_code == 200:
        results = response.json().get("results", [])
        summary = "\n".join([f"- {r['title']}: {r['content']}" for r in results])
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"


print("✅ Tavily search tool ready.")

✅ Tavily search tool ready.


In [10]:
# ── HELPER: run_agent — Replaces OpenAI's Runner.run() ───────────────────────
# OpenAI Agents SDK's Runner.run() automatically handles:
#   1. Sending messages to the model
#   2. Executing tool calls when the model requests them
#   3. Returning the final output
#
# With Gemini, we build this loop ourselves.
# This function handles models WITH tools (tavily_search) and WITHOUT tools.

def run_agent(model, user_input: str, history: list = None) -> str:
    """
    Runs a Gemini agent with automatic function-calling loop.
    Equivalent to: await Runner.run(agent, input, session)

    Args:
        model: The Gemini GenerativeModel.
        user_input: The user's question / input text.
        history: Optional conversation history list for memory.

    Returns:
        str: The agent's final text response.
    """
    if history is None:
        history = []

    # Start a chat session with history for memory
    chat = model.start_chat(history=history)

    # Send the user's message
    response = chat.send_message(user_input)

    # Loop: handle function calls until we get a text response
    max_iterations = 10
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        parts = response.candidates[0].content.parts

        # Check if any part is a function call
        function_call_part = None
        for part in parts:
            if hasattr(part, "function_call") and part.function_call.name:
                function_call_part = part
                break

        if function_call_part:
            fc = function_call_part.function_call
            fn_name = fc.name
            fn_args = dict(fc.args)

            print(f"  🔧 Tool call: {fn_name}({fn_args})")

            # Execute the function
            if fn_name == "tavily_search":
                result = tavily_search(**fn_args)
            else:
                result = f"Unknown function: {fn_name}"

            # Send the function result back to Gemini
            response = chat.send_message(
                genai.protos.Content(
                    parts=[
                        genai.protos.Part(
                            function_response=genai.protos.FunctionResponse(
                                name=fn_name,
                                response={"result": result},
                            )
                        )
                    ]
                )
            )
        else:
            # No function call — we have the final text response
            break

    # Extract all text parts from the final response
    final_text = ""
    for part in response.candidates[0].content.parts:
        if hasattr(part, "text") and part.text:
            final_text += part.text

    # Update history for memory
    history.extend(chat.history)

    return final_text


print("✅ run_agent() helper defined (replaces Runner.run).")

✅ run_agent() helper defined (replaces Runner.run).


# TASK 3. DEFINE TWO AI AGENTS (RESEARCHER & ANALYST)

In [11]:
# ── Original Cell 8: Researcher Agent ────────────────────────────────────────
# ORIGINAL:
#   class AnalysisSummary(BaseModel):
#       summary: str
#
#   researcher_agent = Agent(
#       name="Researcher",
#       instructions="...",
#       model="gpt-4.1-mini",
#       tools=[tavily_search],
#       output_type=AnalysisSummary,
#   )
#
# GEMINI EQUIVALENT:
#   - Agent()             → genai.GenerativeModel()
#   - name=               → included in system_instruction
#   - instructions=       → system_instruction=
#   - model="gpt-4.1-mini"→ model_name=GEMINI_MODEL  ("models/gemini-2.0-flash")
#   - tools=[tavily_search] → tools=[tavily_search]
#   - output_type=AnalysisSummary → enforced via system prompt

# Define Pydantic model for reference (same as original)
class AnalysisSummary(BaseModel):
    summary: str


# Create the Researcher agent
researcher_agent = genai.GenerativeModel(
    model_name=GEMINI_MODEL,                    # "models/gemini-2.0-flash" (replaces gpt-4.1-mini)
    tools=[tavily_search],                      # Same tool registration
    system_instruction="""
## Context
You are a research agent named "Researcher" with access to the tavily_search tool.

## Instruction
Given a user query, use the tavily_search tool to find relevant information and summarize the key findings.

## Input
- A research query from the user.

## Output
- A summary of key findings in a maximum of 5 bullet points.
- Return ONLY the summary text, no extra commentary.
""",
)

print("✅ Researcher AI agent is now ready")

✅ Researcher AI agent is now ready


In [12]:
# ── Original Cell 9: Run the researcher agent ────────────────────────────────
# ORIGINAL:
#   researcher_result = await Runner.run(
#       researcher_agent,
#       "Why is Labubu so popular and what are the rarest Labubu collectibles?"
#   )
#   researcher_result
#
# GEMINI EQUIVALENT:

researcher_result = run_agent(
    researcher_agent,
    "Why is Labubu so popular and what are the rarest Labubu collectibles?"
)

# Show raw result (in original, this displayed the Runner result object)
researcher_result

  🔧 Tool call: tavily_search({'query': 'Labubu popularity and rarest collectibles'})


'* Labubu\'s popularity is driven by its unique design and the growing trend of collecting designer toys, particularly from brands like Pop Mart.\n* The rarest Labubu collectibles are often unique prototypes or limited-edition collaborations, with some fetching extremely high prices.\n* The Life-Size Mint Green Labubu Prototype is valued at $170,000, and a Life-Size Brown Labubu at $131,000.\n* Other rare and valuable items include the "3 Wise Labubu" ($70,000), Sacai x SEVENTEEN x Labubu ($30,000+), and Labubu x Vans Oldskool Monsters Forever ($10,585).\n* Popular series include "The Monsters - Exciting Macaron Vinyl Face Blind Box," "The Monsters Fall in Wild Series - Vinyl Plush Doll," and "The Monsters Coca-Cola Series - Vinyl Face Blind Box."'

In [13]:
# ── Original Cell 10: Display researcher result ──────────────────────────────
# ORIGINAL:
#   print_markdown(f"### 🤖 Agent's Answer\n{researcher_result.final_output.summary}")
#
# GEMINI EQUIVALENT:
#   In the original, researcher_result.final_output.summary accessed the Pydantic model.
#   With Gemini, run_agent() returns a plain string directly.

print_markdown(f"### 🤖 Agent's Answer\n{researcher_result}")

### 🤖 Agent's Answer
* Labubu's popularity is driven by its unique design and the growing trend of collecting designer toys, particularly from brands like Pop Mart.
* The rarest Labubu collectibles are often unique prototypes or limited-edition collaborations, with some fetching extremely high prices.
* The Life-Size Mint Green Labubu Prototype is valued at $170,000, and a Life-Size Brown Labubu at $131,000.
* Other rare and valuable items include the "3 Wise Labubu" ($70,000), Sacai x SEVENTEEN x Labubu ($30,000+), and Labubu x Vans Oldskool Monsters Forever ($10,585).
* Popular series include "The Monsters - Exciting Macaron Vinyl Face Blind Box," "The Monsters Fall in Wild Series - Vinyl Plush Doll," and "The Monsters Coca-Cola Series - Vinyl Face Blind Box."

In [14]:
# ── Original Cell 11: Analyst Agent ──────────────────────────────────────────
# ORIGINAL:
#   analyst_agent = Agent(
#       name="Analyst",
#       instructions="...",
#       model="gpt-4.1-mini",
#       output_type=AnalysisSummary,
#   )
#
# GEMINI EQUIVALENT:
#   - No tools needed for the analyst (it just analyzes text)
#   - output_type enforcement via system prompt

analyst_agent = genai.GenerativeModel(
    model_name=GEMINI_MODEL,                    # "models/gemini-2.0-flash" (replaces gpt-4.1-mini)
    # No tools — analyst only analyzes text passed to it
    system_instruction="""
## Context
You are an analyst named "Analyst" who receives research notes generated by the research agent.

## Instruction
Given the research notes, analyze the content and extract key trends, risks, or insights.

## Input
- Research notes (summaries of findings from the research agent).

## Output
- A concise analysis (no more than 2 paragraphs) highlighting key trends, risks, or insights.
- Return ONLY the analysis text, no extra commentary.
""",
)

print("✅ Analyst AI agent is now ready")

✅ Analyst AI agent is now ready


**PRACTICE OPPORTUNITY:**  

- **Use the `researcher_agent` and `analyst_agent` to analyze a specific research question or topic.**

  - **Provide a research question (e.g., "What are the latest trends in Batteries for Electric Vehicles?").**
  - **Call the `researcher_agent` with the research question to obtain key findings.**
  - **Pass the researcher's summary to the `analyst_agent` to extract key trends, risks, or insights.**
  - **Show how you would structure the conversation and pass information between the agents.**

# TASK 4: DEFINE A WRITER AI AGENT (EXECUTIVE REPORT GENERATOR)

In [15]:
# ── Original Cell 16: Writer Agent ───────────────────────────────────────────
# ORIGINAL:
#   class FinalReport(BaseModel):
#       short_summary: str
#       markdown_report: str
#       follow_up_questions: list[str]
#
#   writer_agent = Agent(
#       name="Writer",
#       instructions="...",
#       model="gpt-4.1-mini",
#       output_type=FinalReport,
#   )
#
# GEMINI EQUIVALENT:
#   - output_type=FinalReport is enforced via Gemini's structured output:
#     generation_config with response_mime_type="application/json" + response_schema
#   - This makes Gemini return valid JSON matching our schema

# Define the Pydantic model (same as original)
class FinalReport(BaseModel):
    short_summary: str               # A brief 2–3 sentence executive summary
    markdown_report: str              # A detailed report in Markdown (at least 500 words)
    follow_up_questions: list[str]    # 3–5 suggested follow-up research questions


# Create the Writer agent with structured JSON output
writer_agent = genai.GenerativeModel(
    model_name=GEMINI_MODEL,                    # "models/gemini-2.0-flash" (replaces gpt-4.1-mini)
    # No tools — writer synthesizes text passed to it
    system_instruction="""
You are a senior market analyst named "Writer" tasked with generating an executive-level research report.

## Context
You will receive:
- The original user query (the research question or topic of interest).
- Summaries and analyses generated by helper agents: 'Researcher' (for bullet-point research findings) and 'Analyst' (for key trends, risks, or insights).

## Instructions
Your job is to synthesize all available information and produce the following outputs:
1. **Executive Summary (short_summary):** Write a concise summary (2-3 sentences) highlighting the most important findings.
2. **Detailed Markdown Report (markdown_report):** Compose a comprehensive, well-structured report in markdown format (at least 500 words) covering key findings, context, implications, and supporting evidence.
3. **Follow-up Research Questions (follow_up_questions):** Suggest 3-5 thoughtful follow-up questions for further investigation.

## Output Format
Return a JSON object with exactly these fields:
- "short_summary": string
- "markdown_report": string
- "follow_up_questions": array of strings
""",
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",     # Force JSON output
        response_schema={                          # Replaces output_type=FinalReport
            "type": "object",
            "properties": {
                "short_summary": {"type": "string"},
                "markdown_report": {"type": "string"},
                "follow_up_questions": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["short_summary", "markdown_report", "follow_up_questions"]
        }
    ),
)

print("✅ Writer AI Agent is now ready")

✅ Writer AI Agent is now ready


**PRACTICE OPPORTUNITY:**
- **Rewrite the `writer` agent's instructions so that it generates all outputs entirely in French (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**
- **Note: We will test this AI Agent in the next task so stay tuned!**

## TASK 5. BUILD A MANAGER FUNCTION TO ORCHESTRATE THE FULL PIPELINE

In [16]:
# ── Original Cell 20: Manager pipeline ───────────────────────────────────────
# ORIGINAL:
#   session = SQLiteSession("research_agent_practice")
#   async def manager_run(user_query: str):
#       researcher_result = await Runner.run(researcher_agent, user_query, session=session)
#       research_summary = researcher_result.final_output.summary
#       analyst_result = await Runner.run(analyst_agent, research_summary, session=session)
#       analysis_summary = analyst_result.final_output.summary
#       ...
#       writer_result = await Runner.run(writer_agent, input_for_writer, session=session)
#       final_output: FinalReport = writer_result.final_output
#       ... display final_output.short_summary, .markdown_report, .follow_up_questions
#
# GEMINI EQUIVALENT:
#   - SQLiteSession → conversation_history list
#   - async def → regular def (Gemini SDK is synchronous)
#   - Runner.run() → run_agent() helper
#   - .final_output.summary → direct string return
#   - Writer returns JSON → we parse it into FinalReport

# Shared conversation history (replaces SQLiteSession)
session_history = []


def manager_run(user_query: str):
    """
    Orchestrates the full multi-agent pipeline:
      Researcher → Analyst → Writer
    Equivalent to the original async manager_run with Runner.run calls.
    """
    # Show the user's question
    print_markdown(f"**User's Request:** {user_query}")

    # ── Step 1: Run the Researcher agent ─────────────────────────────────────
    # ORIGINAL: researcher_result = await Runner.run(researcher_agent, user_query, session=session)
    # ORIGINAL: research_summary = researcher_result.final_output.summary
    print("\n📡 Step 1: Running Researcher agent...")
    research_summary = run_agent(researcher_agent, user_query, session_history)
    print("  ✅ Research complete.\n")

    # ── Step 2: Run the Analyst agent ─────────────────────────────────────────
    # ORIGINAL: analyst_result = await Runner.run(analyst_agent, research_summary, session=session)
    # ORIGINAL: analysis_summary = analyst_result.final_output.summary
    print("🔍 Step 2: Running Analyst agent...")
    analysis_summary = run_agent(analyst_agent, research_summary, session_history)
    print("  ✅ Analysis complete.\n")

    # ── Step 3: Prepare input for the Writer agent ───────────────────────────
    # ORIGINAL: input_for_writer = f"Original query: ...\nResearch summary: ...\nAnalysis summary: ..."
    input_for_writer = (
        f"Original query: {user_query}\n"
        f"Research summary: {research_summary}\n"
        f"Analysis summary: {analysis_summary}"
    )

    # ── Step 4: Run the Writer agent ──────────────────────────────────────────
    # ORIGINAL: writer_result = await Runner.run(writer_agent, input_for_writer, session=session)
    # ORIGINAL: final_output: FinalReport = writer_result.final_output
    print("✍️  Step 3: Running Writer agent...")
    writer_raw = run_agent(writer_agent, input_for_writer, session_history)
    print("  ✅ Report generated.\n")

    # Parse the JSON output into our FinalReport model
    # (Gemini returns JSON string because we set response_mime_type="application/json")
    try:
        final_output = FinalReport(**json.loads(writer_raw))
    except (json.JSONDecodeError, Exception) as e:
        print(f"⚠️ Could not parse structured output: {e}")
        print("Displaying raw output instead:\n")
        print_markdown(writer_raw)
        return

    # ── Step 5: Display results ──────────────────────────────────────────────
    # ORIGINAL: print_markdown(f"### 📝 **Short Summary:**\n{final_output.short_summary}")
    # ORIGINAL: print_markdown(f"### 📄 **Full Report:**\n{final_output.markdown_report}")
    # ORIGINAL: print_markdown("### 🔍 **Follow-Up Questions:**\n- " + "\n- ".join(...))
    print_markdown("---")
    print_markdown(f"### 📝 **Short Summary:**\n{final_output.short_summary}")
    print_markdown("\n\n-----------------\n\n")
    print_markdown(f"### 📄 **Full Report (Markdown):**\n{final_output.markdown_report}")
    print_markdown("\n\n-----------------\n\n")
    print_markdown(
        "### 🔍 **Follow-Up Questions:**\n- "
        + "\n- ".join(final_output.follow_up_questions)
    )
    print_markdown("\n\n-----------------\n\n")


print("✅ manager_run() pipeline defined.")

✅ manager_run() pipeline defined.


In [17]:
# ── Original Cell 21: Run the full pipeline ──────────────────────────────────
# ORIGINAL:
#   await manager_run("What is the public sentiment and expert reviews about the Tesla Cybertruck?")
#
# GEMINI EQUIVALENT:
#   No 'await' needed — Gemini SDK is synchronous

manager_run("What is the public sentiment and expert reviews about the AI Chat models like chatgpt and gemini?")

**User's Request:** What is the public sentiment and expert reviews about the AI Chat models like chatgpt and gemini?


📡 Step 1: Running Researcher agent...
  🔧 Tool call: tavily_search({'query': 'public sentiment and expert reviews about AI Chat models like ChatGPT and Gemini'})
  ✅ Research complete.

🔍 Step 2: Running Analyst agent...
  ✅ Analysis complete.

✍️  Step 3: Running Writer agent...
  ✅ Report generated.



---

### 📝 **Short Summary:**
Public and expert reviews indicate that AI chat models like ChatGPT and Gemini possess distinct strengths and weaknesses, with no single model universally outperforming others. While Gemini is praised for its multimodal capabilities and Claude for instruction following, ChatGPT is noted for conversational nuance. User experiences vary, leading to a dynamic market where individuals seek the best-fit model for specific tasks, suggesting ongoing competition and innovation.



-----------------



### 📄 **Full Report (Markdown):**
# Executive Report: Public Sentiment and Expert Reviews of AI Chat Models (ChatGPT & Gemini)

## Introduction
This report synthesizes public sentiment and expert reviews concerning leading AI chat models, specifically focusing on OpenAI's ChatGPT and Google's Gemini. The landscape of generative AI is rapidly evolving, with these models at the forefront of conversational AI technology. Understanding user perceptions and expert analyses is crucial for developers, businesses, and consumers navigating this increasingly sophisticated technological frontier.

## Key Findings

### Comparative Performance and User Preferences

The prevailing sentiment suggests that while ChatGPT, Gemini, and Claude are all powerful AI models, they exhibit differentiated strengths and weaknesses. This implies that a "one-size-fits-all" AI solution is not yet a reality, and user preference is often task-dependent.

*   **Instruction Following:** Several user reviews highlight Claude as potentially superior to both ChatGPT and Gemini in its ability to precisely follow complex instructions. This suggests that for tasks requiring strict adherence to user directives, Claude may offer a more reliable experience.
*   **Conversational Nuance and Voice Mode:** ChatGPT, on the other hand, is recognized for its adeptness in understanding conversational nuances and specific prompt instructions. This is particularly evident in its voice mode, where it effectively interprets user commands regarding feedback or conversation continuation. Gemini, in contrast, has been noted to falter in these specific voice-based interaction scenarios.
*   **Web Searching and Browsing:** ChatGPT, Gemini, and Claude appear to offer comparable capabilities when it comes to web searching and browsing functionalities. This parity in a core feature suggests that competition in this area is fierce, and users can expect similar utility across these platforms for information retrieval tasks.
*   **Agent Mode:** All three prominent models – ChatGPT, Gemini, and Claude – feature an "agent mode" that allows them to control a user's browser on their behalf. This indicates a shared trend towards enabling more autonomous AI interaction with online environments.

### Gemini: A Promising Multimodal Advancement

Google's Gemini has been identified as a significant advancement in the field of AI, particularly due to its nature as a "large multimodal model." This means Gemini is designed to process and understand various types of data, including text, images, audio, and potentially video, in a more integrated fashion than purely text-based models. While this positions Gemini as a cutting-edge development, direct comparisons indicate that it does not universally outperform ChatGPT across all functionalities. Its strength lies in its potential for handling diverse data inputs, which could unlock new applications and user experiences.

### Perceived Drawbacks and Evolving Standards

User reviews suggest that while all AI models have limitations, ChatGPT is sometimes perceived as having more drawbacks compared to Gemini and Claude. This perception, even with Gemini's own specific advantages, underscores the subjective nature of user experience and the rapidly evolving standards for AI performance. As users become more familiar with the capabilities and limitations of these models, their expectations and critical assessments continue to mature.

## Expert and Public Outlook

Experts and the public alike view the current state of AI chat models as dynamic and competitive. The emergence of models like Gemini signals a strong push towards more sophisticated, multimodal AI. The ongoing development and refinement of these tools suggest that capabilities such as nuanced understanding, precise instruction following, and seamless multimodal integration will continue to improve. The user-driven differentiation highlights the importance of ongoing research and development focused on specific use cases and user needs.

## Conclusion

The AI chat model market, exemplified by ChatGPT and Gemini, is characterized by intense competition and rapid innovation. Users are actively evaluating these models based on their specific strengths, leading to a diverse range of preferences and use cases. While Gemini represents a significant leap in multimodal AI, ChatGPT maintains strengths in conversational understanding, and Claude is noted for its instruction-following prowess. The industry trend points towards greater integration of multimodal capabilities and enhanced accuracy in understanding and executing user commands, promising a future of even more powerful and versatile AI assistants.



-----------------



### 🔍 **Follow-Up Questions:**
- How do the underlying architectures of ChatGPT (GPT-4/3.5) and Gemini (e.g., Gemini Pro, Ultra) differ, and how do these differences translate into their respective strengths and weaknesses?
- What are the specific benchmarks or performance metrics used by experts to compare ChatGPT, Gemini, and Claude, and what are the results for tasks like creative writing, coding assistance, and factual accuracy?
- Beyond the mentioned strengths, what are the most commonly cited limitations or 'drawbacks' of ChatGPT, Gemini, and Claude by both average users and AI professionals?
- How does the training data and methodology for Gemini's multimodal capabilities compare to OpenAI's approaches for handling different data types (e.g., GPT-4V), and what are the implications for their practical applications?
- What is the projected roadmap for future development for ChatGPT and Gemini, particularly concerning improvements in instruction following, conversational memory, and cross-modal reasoning?



-----------------



**PRACTICE OPPORTUNITY:**
- **Run the pipeline again using the `writer` agent with French language report generation capability (from the previous practice opportunity).**

- **Create a new multi-agent team that performs creative advertising:**
  - **Define a `Creative_Director` agent that brainstorms 3–5 ad ideas.**
  - **Define a `Strategist` agent that selects the top 2 ideas and explains the reasoning.**
  - **Define a `Copywriter` agent that writes tweets for each top campaign.**
  - **Set up any necessary tools for these agents.**
  - **Build a pipeline where the output of each agent is passed to the next, similar to the previous example.**
  - **Try running the full pipeline on a sample prompt (e.g., "Launch campaign for a new eco-friendly water bottle in Bali").**